In [ ]:
import cv2
import datetime
import imageio
import models
import numpy as np
import os
import pandas as pd
import random
import torch
import torch.nn.functional as F
from PIL import Images
from IPython.display import HTML, clear_output
from importlib import reload
from matplotlib import animation
import matplotlib.pyplot as plt
import utils
from torchsummary import summary

reload(models)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.set_default_dtype(torch.float32)

# Settings

In [ ]:
config_name = "scr_1seg_koopman"

config = utils.load_config(config_name)

for k, v in config.items():
    print(f"{k}: {v}")

if "1seg" in config["dataset"]:
    n_frames_steady_state=320
elif "2seg" in config["dataset"]:
    n_frames_steady_state=300

date_string=datetime.now().strftime("%Y%m%d_%H%M")
model_string = date_string

model_string+=f"_{config['dataset']}_{config['dynamics_type']}"
if config["use_attention_decoder"]:
    model_string += "_attn"

if config["save_model"]:
    save_dir = os.path.join("results", "models", model_string)
    os.makedirs(save_dir, exist_ok=True)

# Load Dataset

In [ ]:
datasets = {
    "scr_match_1seg": f"scr_match/scr_dataset_raw_1seg_{config['resolution']}x{config['resolution']}_59fps.npz",
    "scr_match_2seg": f"scr_match/scr_dataset_raw_2seg_{config['resolution']}x{config['resolution']}_59fps.npz",
}

dataset_path = f"data/{datasets[config['dataset']]}"

dataset=np.load(dataset_path)

# Prepare Dataloader

In [ ]:
data = np.load(dataset_path, mmap_mode='r')  
o = data['images'] 

if o.max() > 1.0:
    o = o / 255.0

if o.ndim == 3:
    o = o[:, None, :, :]
else:
    if o.shape[-1] == 3 and o.shape[1] != 3:
        o = np.transpose(o, (0, 3, 1, 2))

actuation_dim = config.get("actuation_dim", 2)

if actuation_dim > 0:
    actuator_input_list = []
    for i in range(1, actuation_dim + 1):
        key = f"p{i}"
        if key not in dataset:
            raise KeyError(f"Actuator input '{key}' not found in dataset keys: {list(dataset.keys())}")
        actuator_input = dataset[key]
        tensor_actuator_input = torch.tensor(actuator_input, dtype=torch.float32)
        if tensor_actuator_input.ndim == 1:
            tensor_actuator_input = tensor_actuator_input.unsqueeze(-1)
        actuator_input_list.append(tensor_actuator_input)
    
    tensor_actuator_inputs = torch.cat(actuator_input_list, dim=-1)
    
    num_delays = config.get("num_actuation_delays", 1)
    
    if num_delays > 1:
        print(f"Creating delayed actuation inputs with {num_delays} timesteps (including current)")
        
        n_samples = tensor_actuator_inputs.shape[0]
        actuation_dim_base = config.get("actuation_dim", 0)
        
        delayed_actuator_inputs = torch.zeros(n_samples, actuation_dim_base * num_delays, dtype=torch.float32)
        
        for t in range(n_samples):
            delayed_states = []
            
            for delay in range(num_delays - 1, -1, -1):
                if t - delay < 0:
                    delayed_states.append(tensor_actuator_inputs[0])
                else:
                    delayed_states.append(tensor_actuator_inputs[t - delay])
            
            delayed_actuator_inputs[t] = torch.cat(delayed_states, dim=0)
        
        tensor_actuator_inputs = delayed_actuator_inputs
        print(f"✓ Delayed actuation shape: {tensor_actuator_inputs.shape}")
        print(f"  Original actuation_dim: {actuation_dim_base}")
        print(f"  New effective actuation_dim: {actuation_dim_base * num_delays}")
    else:
        print(f"Using single-timestep actuation (num_actuation_delays=1)")
else:
    print("No actuation inputs (actuation_dim=0) - creating empty actuation tensor")
    n_samples = len(o)
    tensor_actuator_inputs = torch.zeros(n_samples, 0, dtype=torch.float32)

tensor_o = torch.from_numpy(o).float()

n_total = tensor_o.shape[0]
n_train = int(config["train_val_ratio"] * n_total)
n_val = n_total - n_train

train_o = tensor_o[:n_train]
val_o = tensor_o[n_train:]

train_actuator_inputs = tensor_actuator_inputs[:n_train]
val_actuator_inputs = tensor_actuator_inputs[n_train:]

class ObservationWithActuatorInputDataset(torch.utils.data.Dataset):
    def __init__(self, observations, actuator_inputs):
        self.observations = observations
        self.actuator_inputs = actuator_inputs

    def __len__(self):
        return self.observations.shape[0]

    def __getitem__(self, idx):
        obs = self.observations[idx]
        actuator = self.actuator_inputs[idx]
        return obs, actuator

train_dataset = ObservationWithActuatorInputDataset(train_o, train_actuator_inputs)
val_dataset = ObservationWithActuatorInputDataset(val_o, val_actuator_inputs)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=False)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=False)

n_steady_total = min(n_frames_steady_state, len(train_dataset) + len(val_dataset))
n_steady_train = int(config["train_val_ratio"] * n_steady_total)
n_steady_val = n_steady_total - n_steady_train

if n_steady_val < config["batch_size"]:
    n_steady_val = config["batch_size"]
    n_steady_train = max(0, n_steady_total - n_steady_val)

steady_state_train_indices = list(range(n_steady_train))
steady_state_val_indices = list(range(n_steady_train, n_steady_train + n_steady_val))

steady_state_train_indices = [i for i in steady_state_train_indices if i < len(train_dataset)]
steady_state_val_indices = [i for i in steady_state_val_indices if i < len(val_dataset)]

steady_state_train_sampler = torch.utils.data.RandomSampler(steady_state_train_indices, replacement=False)
train_loader_steady_state = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    sampler=steady_state_train_sampler
)

steady_state_val_sampler = torch.utils.data.RandomSampler(steady_state_val_indices, replacement=False)
val_loader_steady_state = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    sampler=steady_state_val_sampler
)

# Initialize Model

In [ ]:
vae = models.VAE(input_channels=o.shape[1], latent_dim=config["latent_dim"],is_vae=config["use_vae"],
use_attention_decoder=config["use_attention_decoder"], attention_feature_dim=config["attention_feature_dim"], attention_downsample_factor=config["attention_downsample_factor"],
background_token_value=config["background_token_value"], image_size=config["resolution"], dynamics_type=config["dynamics_type"],
gumbel_noise_strength=config["gumbel_noise_strength"]).to(device)

dynamics_config = config.copy()
num_delays = config.get("num_actuation_delays", 1)
if num_delays > 1:
    dynamics_config["actuation_dim"] = config["actuation_dim"] * num_delays
    print(f"Dynamics model effective actuation_dim: {dynamics_config['actuation_dim']} (base: {config['actuation_dim']} × {num_delays} delays)")

latent_dynamics = models.create_dynamics_model(
    config=dynamics_config,
    device=device
).to(device)

print(f"Using {config['dynamics_type']} dynamics model")

def split_dynamics_params_no_weight_decay(module):
    decay, no_decay = [], []
    for name, param in module.named_parameters():
        if any(s in name for s in ["m_scale", "k_scale", "d_scale","alpha_raw","beta_raw"]):
            no_decay.append(param)
        else:
            decay.append(param)
    return decay, no_decay

decay_dyn, no_decay_dyn = split_dynamics_params_no_weight_decay(latent_dynamics)

optimizer = torch.optim.AdamW([
    {'params': vae.encoder.parameters(), 'lr': config["lr_encoder"], 'weight_decay': config["weight_decay_encoder"]},
    {'params': vae.decoder.parameters(), 'lr': config["lr_decoder"], 'weight_decay': config["weight_decay_decoder"]},
    {'params': decay_dyn, 'lr': config["lr_dynamics"], 'weight_decay': config["weight_decay_dynamics"]},
    {'params': no_decay_dyn, 'lr': config["lr_dynamics"], 'weight_decay': 0.0}
])

if config["n_epochs_warmup"] > 0:
    for i, param_group in enumerate(optimizer.param_groups):
        if i == 0:
            param_group['lr'] = config["lr_encoder"] * config["warmup_factor"]
        elif i == 2:
            param_group['lr'] = config["lr_dynamics"] * config["warmup_factor"]
        elif i == 3:
            param_group['lr'] = config["lr_dynamics"] * config["warmup_factor"]
        elif i == 1:
            param_group['lr'] = config["lr_decoder"]

class ManualWarmupLRScheduler:
    """Manual LR scheduler with warmup and periodic reduction."""
    def __init__(self, optimizer, base_lrs, warmup_factor, warmup_epochs, patience, factor):
        self.optimizer = optimizer
        assert len(base_lrs) == 4, "base_lrs must be a list or tuple of 4 values"
        self.base_lrs = list(base_lrs)
        self.warmup_factor = warmup_factor
        self.warmup_epochs = warmup_epochs
        self.epoch = 0
        self.patience = patience
        self.factor = factor
        self._last_reduce_epoch = warmup_epochs

    def step(self):
        self.epoch += 1
        if self.epoch <= self.warmup_epochs:
            for i, param_group in enumerate(self.optimizer.param_groups):
                if i == 0 or i == 2 or i == 3:
                    lr = self.base_lrs[i] * (self.warmup_factor + (1.0 - self.warmup_factor) * (self.epoch / self.warmup_epochs))
                    param_group['lr'] = lr
                else:
                    param_group['lr'] = self.base_lrs[i]
        else:
            if (self.epoch - self._last_reduce_epoch) >= self.patience:
                for i, param_group in enumerate(self.optimizer.param_groups):
                    old_lr = param_group['lr']
                    new_lr = max(old_lr * self.factor, 1e-8)
                    param_group['lr'] = new_lr
                self._last_reduce_epoch = self.epoch

    def get_last_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]

manual_scheduler = ManualWarmupLRScheduler(
    optimizer,
    base_lrs=[
        config.get("lr_encoder"),
        config.get("lr_decoder"),
        config.get("lr_dynamics"),
        config.get("lr_dynamics")
    ],
    warmup_factor=config["warmup_factor"],
    warmup_epochs=config["n_epochs_warmup"],
    patience=config.get("lr_scheduler_patience", 10),
    factor=config.get("lr_scheduler_factor", 0.1)
)

loss_df = pd.DataFrame(columns=[
    "lr_enc","lr_dec","lr_dyn","lr_dyn_scale",
    "epoch", 
    "t_total", "t_static_recon", "t_kl", "t_dyn_recon", "t_dyn", "t_steady", "t_dyn_attn", "t_attn_pos",
    "v_total", "v_static_recon", "v_kl", "v_dyn_recon", "v_dyn", "v_steady", "v_dyn_attn", "v_attn_pos"
])


In [ ]:
example_input_shape = (o.shape[1], o.shape[2], o.shape[3]) if len(o.shape) == 4 else (1, 32, 32)
print("VAE parameter summary:")
summary(vae, input_size=example_input_shape, device=str(device));


# Training

In [ ]:
i_epoch=0

In [ ]:
delta_t = torch.tensor(config["delta_t"], device=device)

def get_current_lr(optimizer):
    lrs = []
    for param_group in optimizer.param_groups:
        lrs.append(param_group['lr'])
    if len(lrs) == 1:
        return lrs[0]
    return lrs

for epoch in range(i_epoch, i_epoch + config["epochs"]):
    lam_attn_pos_schedule = list(zip(config["lam_attn_pos_change_epochs"], config["lam_attn_pos"]))
    lam_attn_pos = lam_attn_pos_schedule[0][1]
    for i in range(len(lam_attn_pos_schedule)):
        change_epoch, val = lam_attn_pos_schedule[i]
        if epoch >= change_epoch:
            lam_attn_pos = val
        else:
            break

    progress = min(epoch / config['lam_beta_schedule_epochs'], 1.0)
    lam_beta = config['lam_beta_start'] + (config['lam_beta_end'] - config['lam_beta_start']) * progress

    epoch_stats = np.zeros((2, 8))
    n_samples = np.zeros(2, dtype=int)
    
    for i, (phase, loader, loader_steady_state, train_flag, data_len) in enumerate([
        ("train", train_loader, train_loader_steady_state, True, len(train_dataset)),
        ("val", val_loader, val_loader_steady_state, False, len(val_dataset))
    ]):
        vae.train() if train_flag else vae.eval()
        latent_dynamics.train() if train_flag else latent_dynamics.eval()
        
        for batch in loader:
            o_batch, actuator_inputs = batch
            o_batch = o_batch.to(device)
            actuator_inputs = actuator_inputs.to(device)
            batch_size = o_batch.size(0)
            
            with torch.set_grad_enabled(train_flag):
                if config["use_vae"]:
                    recon, z_sample, mu, logvar = vae(o_batch)
                else:
                    recon, mu = vae(o_batch)
                z = mu
                if "harmonic" in config["dynamics_type"]:
                    o_batch_steady, actuator_steady = next(iter(loader_steady_state))
                    o_batch_steady = o_batch_steady.to(device)
                    actuator_steady = actuator_steady.to(device)

                    if config["use_vae"]:
                        _, _, z_steady, _ = vae(o_batch_steady)
                    else:
                        _, z_steady = vae(o_batch_steady)
                    o_dot_steady = torch.zeros_like(o_batch_steady)
                    z_dot_steady = vae.latent_velocity_from_observation_velocity(o_batch_steady, o_dot_steady)
                    
                    x0_expanded = latent_dynamics.osc_net.x0.unsqueeze(0).expand(z_steady.shape[0], -1)
                    loss_ss1 = F.mse_loss(z_steady, x0_expanded)
                    loss_ss2 = F.mse_loss(z_dot_steady * delta_t, torch.zeros_like(z_dot_steady))
                    
                    z_steady_next, z_dot_steady_next = latent_dynamics.forward(
                        z_steady, z_dot_steady, actuator_steady, config["delta_t"]
                    )
                    loss_ss3 = F.mse_loss(z_steady_next, x0_expanded)
                    loss_ss4 = F.mse_loss(z_dot_steady_next * delta_t, torch.zeros_like(z_dot_steady_next))
                    loss_steady_total = loss_ss1 + loss_ss2 + loss_ss3 + loss_ss4
                else:
                    loss_steady_total = torch.tensor(0.0, device=device)
                
                o_dot = torch.zeros_like(o_batch)
                o_dot[0] = (o_batch[1] - o_batch[0]) / delta_t
                o_dot[1:-1] = (o_batch[2:] - o_batch[:-2]) / (2.0 * delta_t)
                o_dot[-1] = (o_batch[-1] - o_batch[-2]) / delta_t
                
                z_dot = vae.latent_velocity_from_observation_velocity(o_batch, o_dot)
                
                if config["use_attention_decoder"]:
                    attention_weights, background_weights, attention_peaks = vae.decoder.get_attention_weights(
                        mu, return_background_weights=True, return_peak_location=True
                    )
                    
                    if config["dynamics_type"] == "harmonic2d_proj":
                        oscillator_position, oscillator_velocity = vae.decoder.project_oscillator_position_and_velocity(mu, z_dot)
                    else:
                        oscillator_position = mu
                        oscillator_velocity = z_dot
                    
                    attn_dot, background_dot = models.attention_velocity_from_observation_velocity(vae, o_batch, o_dot)

                    if config["lam_attn"]>0.0:
                        attention_consistency_loss = models.attention_consistency_loss_via_velocity(
                            o_dot, attn_dot, background_dot
                        )
                    else:
                        attention_consistency_loss = torch.tensor(0.0, device=device)
                    
                    if config["dynamics_type"] == "harmonic2d_proj":
                        attention_position_loss = models.attention_position_loss(
                            oscillator_position, oscillator_velocity, attention_weights, attn_dot, delta_t
                        )
                    elif config["dynamics_type"] == "harmonic2d_vel":
                        attention_position_loss = models.attention_velocity_loss(
                            oscillator_position, oscillator_velocity, attention_weights, attn_dot, delta_t
                        )
                    else:
                        attention_position_loss = torch.tensor(0.0, device=device)
                else:
                    attention_consistency_loss = torch.tensor(0.0, device=device)
                    attention_position_loss = torch.tensor(0.0, device=device)
                
                if config["use_vae"]:
                    recon_loss_static, kl_loss = models.vae_loss(recon, o_batch, mu, logvar)
                else:
                    recon_loss_static = F.mse_loss(recon, o_batch)
                    kl_loss = torch.tensor(0.0, device=device)
                recon_loss_dyn, dyn_loss = latent_dynamics.ldm_loss(
                    o_batch, z, z_dot, actuator_inputs, vae, config["delta_t"]
                )
                
                loss = (
                    recon_loss_static
                    + lam_beta * kl_loss
                    + config["lam_o"] * recon_loss_dyn
                    + config["lam_z"] * dyn_loss
                    + config["lam_steady"] * loss_steady_total
                    + config["lam_attn"] * attention_consistency_loss
                    + lam_attn_pos * attention_position_loss
                )
                
                if train_flag:
                    optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(
                       list(vae.parameters()) + list(latent_dynamics.parameters()), 
                       max_norm=1.0
                    )
                    optimizer.step()

            epoch_stats[i] += np.array([
                loss.item(),
                recon_loss_static.item(),
                kl_loss.item(),
                recon_loss_dyn.item(),
                dyn_loss.item(),
                loss_steady_total.item(),
                attention_consistency_loss.item(),
                attention_position_loss.item()
            ])
            n_samples[i] += batch_size

    avg = epoch_stats / n_samples[:, None]
    current_lr = get_current_lr(optimizer)
    loss_df.loc[epoch] = [
        *current_lr,
        epoch + 1,
        avg[0, 0], avg[0, 1], avg[0, 2], avg[0, 3], avg[0, 4], avg[0, 5], avg[0, 6], avg[0, 7],
        avg[1, 0], avg[1, 1], avg[1, 2], avg[1, 3], avg[1, 4], avg[1, 5], avg[1, 6], avg[1, 7]
    ]
    clear_output()
    print(loss_df.tail(10).to_string(line_width=1000))
    i_epoch = epoch + 1

    manual_scheduler.step()

    if config["save_model"] and (epoch + 1) % config["save_frequency"] == 0:
        utils.save_model_and_results(
            save_dir=save_dir,
            vae=vae,
            latent_dynamics=latent_dynamics,
            config=config,
            loss_df=loss_df,
            epoch=i_epoch
        )

# Quick evaluation

## Visualizing oscillator matrices

In [ ]:
if "harmonic" in config.get("dynamics_type", ""):
    M_inv, K, D = latent_dynamics.osc_net.give_Minv_KD()

    if config.get("harmonic_use_nonlinear_forcing", False):
        W = latent_dynamics.osc_net.W_raw
        b = latent_dynamics.osc_net.b_raw

    def plot_matrix(mat, title, ax=None, cmap="viridis"):
        if ax is None:
            fig, ax = plt.subplots()
        im = ax.imshow(mat.cpu().detach().numpy(), cmap=cmap, aspect="auto")
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    plot_matrix(M_inv, "Mass Matrix (M^-1)", ax=axs[0], cmap="Blues")
    plot_matrix(K, "Stiffness Matrix (K)", ax=axs[1], cmap="Reds")
    plot_matrix(D, "Damping Matrix (D)", ax=axs[2], cmap="Greens")
    plt.tight_layout()
    plt.show()


## Losses over epochs

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(16, 8))
fig.suptitle("Training and Validation Losses", fontsize=16)

axs[0, 0].plot(loss_df["epoch"], loss_df["t_total"], label="Train")
axs[0, 0].plot(loss_df["epoch"], loss_df["v_total"], label="Val")
axs[0, 0].set_title("Total Loss")
axs[0, 0].set_xlabel("Epoch")
axs[0, 0].set_ylabel("Loss")
axs[0, 0].set_yscale("log")
axs[0, 0].legend()

axs[0, 1].plot(loss_df["epoch"], loss_df["t_static_recon"], label="Train")
axs[0, 1].plot(loss_df["epoch"], loss_df["v_static_recon"], label="Val")
axs[0, 1].set_title("Static Recon Loss")
axs[0, 1].set_xlabel("Epoch")
axs[0, 1].set_ylabel("Loss")
axs[0, 1].set_yscale("log")
axs[0, 1].legend()

axs[0, 2].plot(loss_df["epoch"], loss_df["t_kl"], label="Train")
axs[0, 2].plot(loss_df["epoch"], loss_df["v_kl"], label="Val")
axs[0, 2].set_title("KL Loss")
axs[0, 2].set_xlabel("Epoch")
axs[0, 2].set_ylabel("Loss")
axs[0, 2].set_yscale("log")
axs[0, 2].legend()

axs[1, 0].plot(loss_df["epoch"], loss_df["t_dyn_recon"], label="Train")
axs[1, 0].plot(loss_df["epoch"], loss_df["v_dyn_recon"], label="Val")
axs[1, 0].set_title("Dynamic Recon Loss")
axs[1, 0].set_xlabel("Epoch")
axs[1, 0].set_ylabel("Loss")
axs[1, 0].set_yscale("log")
axs[1, 0].legend()

axs[1, 1].plot(loss_df["epoch"], loss_df["t_dyn"], label="Train")
axs[1, 1].plot(loss_df["epoch"], loss_df["v_dyn"], label="Val")
axs[1, 1].set_title("Dynamic Loss")
axs[1, 1].set_xlabel("Epoch")
axs[1, 1].set_ylabel("Loss")
axs[1, 1].set_yscale("log")
axs[1, 1].legend()

axs[1, 2].plot(loss_df["epoch"], loss_df["t_dyn_attn"], label="Train")
axs[1, 2].plot(loss_df["epoch"], loss_df["v_dyn_attn"], label="Val")
axs[1, 2].set_title("Attention Consistency Loss")
axs[1, 2].set_xlabel("Epoch")
axs[1, 2].set_ylabel("Loss")
axs[1, 2].set_yscale("log")
axs[1, 2].legend()

axs[2, 0].plot(loss_df["epoch"], loss_df["t_attn_pos"], label="Train")
axs[2, 0].plot(loss_df["epoch"], loss_df["v_attn_pos"], label="Val")
axs[2, 0].set_title("Attention Position Loss")
axs[2, 0].set_xlabel("Epoch")
axs[2, 0].set_ylabel("Loss")
axs[2, 0].set_yscale("log")
axs[2, 0].legend()

axs[2, 1].plot(loss_df["epoch"], loss_df["t_steady"], label="Train")
axs[2, 1].plot(loss_df["epoch"], loss_df["v_steady"], label="Val")
axs[2, 1].set_title("Steady State Loss")
axs[2, 1].set_xlabel("Epoch")
axs[2, 1].set_ylabel("Loss")
axs[2, 1].set_yscale("log")
axs[2, 1].legend()


plt.tight_layout()
plt.show()


## Sequence reconstruction

In [ ]:
vae.eval()
latent_dynamics.eval()

num_consecutive_batches = 4

val_batches = list(val_loader)
max_start = len(val_batches) - num_consecutive_batches
start_batch_idx = np.random.randint(0, max_start + 1)

obs_list = []
act_list = []
for i in range(num_consecutive_batches):
    obs, act = val_batches[start_batch_idx + i]
    obs_list.append(obs)
    act_list.append(act)

o_batch = torch.cat(obs_list, dim=0)
actuator_inputs = torch.cat(act_list, dim=0)
o_batch = o_batch.to(device)
actuator_inputs = actuator_inputs.to(device)

n = min(8, o_batch.size(0))

start_idx = np.random.randint(0, o_batch.size(0) - n + 1)
o_seq = o_batch[start_idx:start_idx + n]
actuator_seq = actuator_inputs[start_idx:start_idx + n]

with torch.no_grad():
    if config["use_vae"]:
        _, _, mu_seq, logvar_seq = vae(o_seq)
    else:
        _, mu_seq = vae(o_seq)

    delta_t = torch.tensor(config["delta_t"], device=o_seq.device, dtype=o_seq.dtype)
    o_dot0 = (o_seq[1:2] - o_seq[0:1]) / delta_t
    
    z_dot0 = vae.latent_velocity_from_observation_velocity(o_seq[0:1], o_dot0)

    z_current = mu_seq[0:1]
    z_dot_current = z_dot0

    z_pred_seq = [z_current]
    z_dot_pred_seq = [z_dot_current]
    
    for t in range(n-1):
        u_t = actuator_seq[t].unsqueeze(0) if actuator_seq[t].dim() == 1 else actuator_seq[t]
        
        if z_current.dim() == 1:
            z_current = z_current.unsqueeze(0)
        if z_dot_current.dim() == 1:
            z_dot_current = z_dot_current.unsqueeze(0)
            
        z_next, z_dot_next = latent_dynamics(z_current, z_dot_current, u_t, dt=config["delta_t"])
        
        z_current = z_next
        z_dot_current = z_dot_next
        
        z_pred_seq.append(z_next)
        z_dot_pred_seq.append(z_dot_next)

z_pred_seq = torch.cat(z_pred_seq, dim=0)
recon_o_seq = vae.decode(z_pred_seq)

orig_o = o_seq.cpu().numpy()
recon_o = recon_o_seq.cpu().numpy()

fig, axes = plt.subplots(2, n, figsize=(n*2, 5))

fig.suptitle('Original vs Predicted Reconstructions', fontsize=16, y=1.05)

for i in range(n):
    if i == n // 2:
        axes[0, i].set_title("Original", fontsize=14, pad=20)
        axes[1, i].set_title("Prediction", fontsize=14, pad=20)
    else:
        axes[0, i].set_title("")
        axes[1, i].set_title("")

axes[0, 0].set_ylabel('Original', fontsize=14, rotation=0, labelpad=50, va='center')
axes[1, 0].set_ylabel('Predicted\nRecon', fontsize=14, rotation=0, labelpad=30, va='center')

is_color = orig_o.shape[1] == 3 or orig_o.shape[1] == 4

for i in range(n):
    if is_color:
        img = np.transpose(orig_o[i], (1, 2, 0))
        if img.shape[2] == 4:
            img = img[..., :3]
        axes[0, i].imshow(img)
    else:
        axes[0, i].imshow(orig_o[i][0], cmap='gray')
    axes[0, i].axis('off')
    
    if is_color:
        img = np.transpose(recon_o[i], (1, 2, 0))
        if img.shape[2] == 4:
            img = img[..., :3]
        axes[1, i].imshow(img)
    else:
        axes[1, i].imshow(recon_o[i][0], cmap='gray')
    axes[1, i].axis('off')

plt.show()

## Correlation matrix

In [ ]:
num_samples = 512 

all_obs = []
for batch in val_loader:
    obs = batch[0]
    all_obs.append(obs)
all_obs = torch.cat(all_obs, dim=0)

rand_indices = np.random.choice(all_obs.shape[0], size=min(num_samples, all_obs.shape[0]), replace=False)
sampled_obs = all_obs[rand_indices].to(device)

vae.eval()
with torch.no_grad():
    _, _, mu_samples, _ = vae(sampled_obs)

mu_np = mu_samples.cpu().numpy()
latent_dim = mu_np.shape[1]

corr_matrix = np.corrcoef(mu_np, rowvar=False)

plt.figure(figsize=(8, 6))
im = plt.imshow(corr_matrix, cmap='bwr', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title("Latent Dimension Correlation Matrix (Random States from Dataset)")
plt.xlabel("Latent Dimension")
plt.ylabel("Latent Dimension")
plt.xticks(range(latent_dim))
plt.yticks(range(latent_dim))
plt.tight_layout()
plt.show()

off_diag = corr_matrix[~np.eye(latent_dim, dtype=bool)]
mean_abs_corr = np.mean(np.abs(off_diag))
print(f"Mean absolute off-diagonal correlation: {mean_abs_corr:.4f}")


## Attention + background maps

In [ ]:
attention_maps, background_maps, peak_location = vae.decoder.get_attention_weights(mu_seq, return_background_weights=True, return_peak_location=True)

if config["dynamics_type"]=="harmonic2d_proj":
    oscillator_position = vae.decoder.project_oscillator_position_and_velocity(mu_seq)
elif config["dynamics_type"]=="harmonic2d_vel":
    oscillator_position = mu_seq

rand_idx = random.randint(0, o_seq.shape[0] - 1)
print(f"Random o_seq[{rand_idx}] shape:", o_seq[rand_idx].shape)

img = np.transpose(o_seq[rand_idx].cpu().numpy(), (1, 2, 0))
if img.shape[2] == 4:
    img = img[..., :3]

attn = attention_maps[rand_idx]
bg = background_maps[rand_idx]
com = peak_location[rand_idx]
if "harmonic2d" in config["dynamics_type"]: 
    osc_pos = oscillator_position[rand_idx]
num_maps = attn.shape[0]
num_bg_maps = bg.shape[0]

total_maps = num_maps + num_bg_maps

fig, axes = plt.subplots(1, total_maps, figsize=(total_maps * 2.5, 2.5))
if total_maps == 1:
    axes = [axes]
fig.suptitle(f'Attention and Background Overlays for o_seq[{rand_idx}]', fontsize=16, y=1.15)

img_min, img_max = img.min(), img.max()
img_norm = (img - img_min) / (img_max - img_min + 1e-8) if img_max > img_min else np.zeros_like(img)
alpha = 0.5

img_h, img_w = img.shape[:2]

if isinstance(com, torch.Tensor):
    com = com.detach().cpu().numpy()
if "harmonic2d" in config["dynamics_type"]:
    if isinstance(osc_pos, torch.Tensor):
        osc_pos = osc_pos.detach().cpu().numpy()

for i in range(num_maps):
    ax = axes[i]
    attn_map = attn[i].detach().cpu().numpy()
    attn_min, attn_max = attn_map.min(), attn_map.max()
    print(f"Attention map {i+1} min: {attn_min:.4f}, max: {attn_max:.4f}")
    if attn_max > attn_min:
        attn_map = (attn_map - attn_min) / (attn_max - attn_min)
    else:
        attn_map = np.zeros_like(attn_map)

    if attn_map.shape != img.shape[:2]:
        attn_map_resized = np.array(Image.fromarray((attn_map * 255).astype(np.uint8)).resize((img.shape[1], img.shape[0]), resample=Image.BILINEAR)) / 255.0
    else:
        attn_map_resized = attn_map
    overlay = np.zeros_like(img_norm)
    overlay[..., 0] = attn_map_resized
    blended = img_norm * (1 - alpha * attn_map_resized[..., None]) + overlay * (alpha * attn_map_resized[..., None])
    ax.imshow(blended)
    ax.set_title(f'Attention Map {i+1}', fontsize=8)
    ax.axis('off')

    if com is not None and i < com.shape[0]:
        y_norm, x_norm = com[i]
        y_pix = ((y_norm + 1) / 2) * (img_h - 1)
        x_pix = ((x_norm + 1) / 2) * (img_w - 1)
        ax.plot(x_pix, y_pix, 'ro', markersize=8, markeredgecolor='k', markeredgewidth=1.0, alpha=0.8)

    if config["dynamics_type"]=="harmonic2d_proj":
        if osc_pos is not None and i < osc_pos.shape[0]:
            y_osc, x_osc = osc_pos[i]
            y_osc_pix = ((y_osc + 1) / 2) * (img_h - 1)
            x_osc_pix = ((x_osc + 1) / 2) * (img_w - 1)
            ax.plot(x_osc_pix, y_osc_pix, 'bo', markersize=8, markeredgecolor='k', markeredgewidth=1.0, alpha=0.8)

for j in range(num_bg_maps):
    ax = axes[num_maps + j]
    bg_map = bg[j].detach().cpu().numpy()
    bg_min, bg_max = bg_map.min(), bg_map.max()
    print(f"Background map {j+1} min: {bg_min:.4f}, max: {bg_max:.4f}")
    if bg_max > bg_min:
        bg_map = (bg_map - bg_min) / (bg_max - bg_min)
    else:
        bg_map = np.zeros_like(bg_map)

    if bg_map.shape != img.shape[:2]:
        bg_map_resized = np.array(Image.fromarray((bg_map * 255).astype(np.uint8)).resize((img.shape[1], img.shape[0]), resample=Image.BILINEAR)) / 255.0
    else:
        bg_map_resized = bg_map
    overlay = np.zeros_like(img_norm)
    overlay[..., 2] = bg_map_resized
    blended = img_norm * (1 - alpha * bg_map_resized[..., None]) + overlay * (alpha * bg_map_resized[..., None])
    ax.imshow(blended)
    ax.set_title(f'Background Map {j+1}', fontsize=8)
    ax.axis('off')

    idx = num_maps + j
    if com is not None and idx < com.shape[0]:
        y_norm, x_norm = com[idx]
        y_pix = ((y_norm + 1) / 2) * (img_h - 1)
        x_pix = ((x_norm + 1) / 2) * (img_w - 1)
        ax.plot(x_pix, y_pix, 'bo', markersize=8, markeredgecolor='k', markeredgewidth=1.0, alpha=0.8)

plt.tight_layout()
plt.show()
import os

plot_dir = "plot"
plot_path = os.path.join(plot_dir, "attention_maps.png")
plt.savefig(plot_path, bbox_inches="tight", dpi=150)
print(f"Figure saved to {plot_path}")


## Latent dim analysis

In [ ]:
state_idx = random.randint(0, mu_seq.shape[0] - 1)
z_base = mu_seq[state_idx].detach().cpu().clone()
latent_dim = z_base.shape[0]

z_range = np.arange(-2, 2.01, 0.25)
n_steps = len(z_range)

fig, axes = plt.subplots(latent_dim, n_steps, figsize=(n_steps * 1.5, latent_dim * 1.5))
if latent_dim == 1:
    axes = axes[None, :]

for i in range(latent_dim):
    for j, z_val in enumerate(z_range):
        z = z_base.clone()
        z[i] = z_val
        z_input = z.unsqueeze(0).to(next(vae.parameters()).device)
        with torch.no_grad():
            recon = vae.decode(z_input)
        recon_img = recon[0].detach().cpu().numpy()
        recon_img = np.transpose(recon_img, (1, 2, 0))
        if recon_img.shape[2] == 4:
            recon_img = recon_img[..., :3]
        img_min, img_max = recon_img.min(), recon_img.max()
        img_norm = (recon_img - img_min) / (img_max - img_min + 1e-8) if img_max > img_min else np.zeros_like(recon_img)
        ax = axes[i, j]
        ax.imshow(img_norm)
        if i == 0:
            ax.set_title(f"{z_val:.2f}", fontsize=8)
        if j == 0:
            ax.set_ylabel(f"z[{i}]", fontsize=8)
        ax.axis('off')

fig.suptitle(f"Effect of Varying Each Latent Dimension (state idx {state_idx})", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## Animations

### Attention overlay gif

In [ ]:

gif_n = min(200, o_batch.size(0))

gif_start_idx = np.random.randint(0, o_batch.size(0) - gif_n + 1)
gif_o_seq = o_batch[gif_start_idx:gif_start_idx + gif_n]
gif_actuator_seq = actuator_inputs[gif_start_idx:gif_start_idx + gif_n]

with torch.no_grad():
    _, _, mu_seq, logvar_seq = vae(gif_o_seq)

    delta_t = torch.tensor(config["delta_t"], device=gif_o_seq.device, dtype=gif_o_seq.dtype)
    o_dot0 = (gif_o_seq[1:2] - gif_o_seq[0:1]) / delta_t
    z_dot0 = vae.latent_velocity_from_observation_velocity(gif_o_seq[0:1], o_dot0)

    z0 = mu_seq[0:1]
    z_dot = z_dot0

    gif_z_pred_seq = [z0.squeeze(0)]
    gif_z_dot_pred_seq = [z_dot.squeeze(0)]
    for t in range(gif_n-1):
        z_next, z_dot_next = latent_dynamics(
            gif_z_pred_seq[-1].unsqueeze(0),
            gif_z_dot_pred_seq[-1].unsqueeze(0),
            gif_actuator_seq[t].unsqueeze(0),
            dt=config["delta_t"]
        )
        gif_z_pred_seq.append(z_next.squeeze(0))
        gif_z_dot_pred_seq.append(z_dot_next.squeeze(0))

    gif_z_pred_seq = torch.stack(gif_z_pred_seq, dim=0)
    gif_z_dot_pred_seq = torch.stack(gif_z_dot_pred_seq, dim=0)

    gif_recon_o_seq = vae.decode(gif_z_pred_seq)
    gif_orig_o = gif_o_seq.cpu().numpy()
    gif_recon_o = gif_recon_o_seq.cpu().numpy()
    gif_attention_maps, gif_background_maps, gif_peak_location = vae.decoder.get_attention_weights(
        mu_seq, return_background_weights=True, return_peak_location=True
    )
    if config["dynamics_type"]=="harmonic2d_proj":
        gif_oscillator_position = vae.decoder.project_oscillator_position_and_velocity(mu_seq)
    elif config["dynamics_type"]=="harmonic2d_vel":
        gif_oscillator_position = mu_seq

is_color = gif_orig_o.shape[1] == 3 or gif_orig_o.shape[1] == 4

num_maps = gif_attention_maps.shape[1]
num_bg_maps = gif_background_maps.shape[1]
alpha = 0.5

frames = []
for i in range(gif_n):
    if is_color:
        orig_img = np.transpose(gif_orig_o[i], (1, 2, 0))
        recon_img = np.transpose(gif_recon_o[i], (1, 2, 0))
        if orig_img.shape[2] == 4:
            orig_img = orig_img[..., :3]
        if recon_img.shape[2] == 4:
            recon_img = recon_img[..., :3]
        orig_img_norm = (orig_img - orig_img.min()) / (orig_img.max() - orig_img.min() + 1e-8) if orig_img.max() > orig_img.min() else np.zeros_like(orig_img)
        recon_img_norm = (recon_img - recon_img.min()) / (recon_img.max() - recon_img.min() + 1e-8) if recon_img.max() > recon_img.min() else np.zeros_like(recon_img)
    else:
        orig_img = gif_orig_o[i][0]
        recon_img = gif_recon_o[i][0]
        orig_img_norm = (orig_img - orig_img.min()) / (orig_img.max() - orig_img.min() + 1e-8) if orig_img.max() > orig_img.min() else np.zeros_like(orig_img)
        recon_img_norm = (recon_img - recon_img.min()) / (recon_img.max() - recon_img.min() + 1e-8) if recon_img.max() > recon_img.min() else np.zeros_like(recon_img)
        orig_img_norm = np.stack([orig_img_norm]*3, axis=-1)
        recon_img_norm = np.stack([recon_img_norm]*3, axis=-1)

    overlays = []
    img_h, img_w = orig_img_norm.shape[:2]

    com = gif_peak_location[i]
    osc_pos = gif_oscillator_position[i]
    if isinstance(com, torch.Tensor):
        com = com.detach().cpu().numpy()
    if isinstance(osc_pos, torch.Tensor):
        osc_pos = osc_pos.detach().cpu().numpy()

    for j in range(num_maps):
        attn_map = gif_attention_maps[i, j].detach().cpu().numpy()
        attn_min, attn_max = attn_map.min(), attn_map.max()
        if attn_max > attn_min:
            attn_map = (attn_map - attn_min) / (attn_max - attn_min)
        else:
            attn_map = np.zeros_like(attn_map)
        if attn_map.shape != orig_img_norm.shape[:2]:
            attn_map_resized = np.array(
                Image.fromarray((attn_map * 255).astype(np.uint8)).resize(
                    (orig_img_norm.shape[1], orig_img_norm.shape[0]), resample=Image.BILINEAR
                )
            ) / 255.0
        else:
            attn_map_resized = attn_map
        overlay = np.zeros_like(orig_img_norm)
        overlay[..., 0] = attn_map_resized
        blended = orig_img_norm * (1 - alpha * attn_map_resized[..., None]) + overlay * (alpha * attn_map_resized[..., None])
        blended_uint8 = (blended * 255).clip(0,255).astype('uint8')

        if com is not None and j < com.shape[0]:
            y_norm, x_norm = com[j]
            y_pix = ((y_norm + 1) / 2) * (img_h - 1)
            x_pix = ((x_norm + 1) / 2) * (img_w - 1)
            pil_img = Image.fromarray(blended_uint8)
            from PIL import ImageDraw
            draw = ImageDraw.Draw(pil_img)
            r = 3
            draw.ellipse([(x_pix - r, y_pix - r), (x_pix + r, y_pix + r)], outline=(255,0,0), width=2)
            blended_uint8 = np.array(pil_img)

        if config["dynamics_type"]=="harmonic2d_proj":
            if osc_pos is not None and j < osc_pos.shape[0]:
                y_osc, x_osc = osc_pos[j]
                y_osc_pix = ((y_osc + 1) / 2) * (img_h - 1)
                x_osc_pix = ((x_osc + 1) / 2) * (img_w - 1)
                pil_img = Image.fromarray(blended_uint8)
                from PIL import ImageDraw
                draw = ImageDraw.Draw(pil_img)
                r = 3
                draw.ellipse([(x_osc_pix - r, y_osc_pix - r), (x_osc_pix + r, y_osc_pix + r)], outline=(0,0,255), width=2)
                blended_uint8 = np.array(pil_img)

        overlays.append(blended_uint8)

    for j in range(num_bg_maps):
        bg_map = gif_background_maps[i, j].detach().cpu().numpy()
        bg_min, bg_max = bg_map.min(), bg_map.max()
        if bg_max > bg_min:
            bg_map = (bg_map - bg_min) / (bg_max - bg_min)
        else:
            bg_map = np.zeros_like(bg_map)
        if bg_map.shape != orig_img_norm.shape[:2]:
            bg_map_resized = np.array(
                Image.fromarray((bg_map * 255).astype(np.uint8)).resize(
                    (orig_img_norm.shape[1], orig_img_norm.shape[0]), resample=Image.BILINEAR
                )
            ) / 255.0
        else:
            bg_map_resized = bg_map
        overlay = np.zeros_like(orig_img_norm)
        overlay[..., 2] = bg_map_resized
        blended = orig_img_norm * (1 - alpha * bg_map_resized[..., None]) + overlay * (alpha * bg_map_resized[..., None])
        blended_uint8 = (blended * 255).clip(0,255).astype('uint8')

        idx = num_maps + j
        if com is not None and idx < com.shape[0]:
            y_norm, x_norm = com[idx]
            y_pix = ((y_norm + 1) / 2) * (img_h - 1)
            x_pix = ((x_norm + 1) / 2) * (img_w - 1)
            pil_img = Image.fromarray(blended_uint8)
            from PIL import ImageDraw
            draw = ImageDraw.Draw(pil_img)
            r = 3
            draw.ellipse([(x_pix - r, y_pix - r), (x_pix + r, y_pix + r)], outline=(255,0,0), width=2)
            blended_uint8 = np.array(pil_img)

        overlays.append(blended_uint8)

    overlays_concat = np.concatenate(overlays, axis=1)

    pil_img = Image.fromarray(overlays_concat)
    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(pil_img)
    text = f"{i+1}"
    try:
        font = ImageFont.truetype("arial.ttf", 12)
    except:
        font = ImageFont.load_default()
    draw.text((2, 2), text, fill=(0,255,0), font=font)
    overlays_concat_with_text = np.array(pil_img)

    frames.append(overlays_concat_with_text)

imageio.mimsave('results/animations/attention_overlays.gif', frames, duration=1/25, loop=0)
print("GIF saved as 'attention_overlays.gif'.")


### Oscillator positions overlay

In [ ]:
# Number of consecutive states to show in the animation (e.g., 24, up to 200)
gif_n = min(200, o_batch.size(0))

# Choose a random starting point for the sequence
gif_start_idx = np.random.randint(0, o_batch.size(0) - gif_n + 1)
gif_o_seq = o_batch[gif_start_idx:gif_start_idx + gif_n]
gif_actuator_seq = actuator_inputs[gif_start_idx:gif_start_idx + gif_n]

with torch.no_grad():
    # Encode all observations to get their latent means (mu)
    _, _, mu_seq, logvar_seq = vae(gif_o_seq)

    # For this visualization, use the true latent means (mu_seq) for oscillator positions,
    # not the predicted rollout, to avoid drift/accumulation artifacts.
    # (If you want to show predicted positions, use gif_z_pred_seq instead.)
    gif_z_for_positions = mu_seq

    # For comparison, get the original observations as numpy
    gif_orig_o = gif_o_seq.cpu().numpy()

    # Get oscillator positions for the sequence (using true mu_seq, not rollout)
    if config["dynamics_type"]=="harmonic2d_proj":
        gif_oscillator_positions = vae.decoder.project_oscillator_position_and_velocity(gif_z_for_positions)  # shape: (gif_n, num_osc, 2)
    elif config["dynamics_type"]=="harmonic2d_vel":
        gif_oscillator_positions = vae.decoder.get_attention_weights(gif_z_for_positions,return_peak_location=True)[1]

# Determine if color or grayscale
is_color = gif_orig_o.shape[1] == 3 or gif_orig_o.shape[1] == 4

frames = []
for i in range(gif_n):
    if is_color:
        # (C, H, W) -> (H, W, C)
        img = np.transpose(gif_orig_o[i], (1, 2, 0))
        if img.shape[2] == 4:
            img = img[..., :3]
    else:
        img = gif_orig_o[i][0]
        img = np.stack([img]*3, axis=-1)
    img_min, img_max = img.min(), img.max()
    img_norm = (img - img_min) / (img_max - img_min + 1e-8) if img_max > img_min else np.zeros_like(img)
    img_h, img_w = img.shape[:2]

    # Plot with matplotlib and draw oscillator positions as blue dots
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img_norm, alpha=0.9)

    # Get oscillator positions for this frame
    pos = gif_oscillator_positions[i]
    if isinstance(pos, torch.Tensor):
        pos = pos.detach().cpu().numpy()
    # pos shape: (num_osc, 2), values in [-1, 1]
    for j in range(pos.shape[0]):
        y_norm, x_norm = pos[j]
        y_pix = ((y_norm + 1) / 2) * (img_h - 1)
        x_pix = ((x_norm + 1) / 2) * (img_w - 1)
        ax.plot(
            float(x_pix), float(y_pix), 'o', color='blue', markersize=10, markeredgecolor='k', markeredgewidth=1.0, alpha=0.9
        )
    ax.set_title(f'Oscillator Positions Overlay\nFrame {i+1}/{gif_n}', fontsize=12)
    ax.axis('off')
    plt.tight_layout()

    # Convert the matplotlib figure to a numpy array for GIF
    fig.canvas.draw()
    frame = np.frombuffer(fig.canvas.get_renderer().buffer_rgba(), dtype=np.uint8)
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    # Convert RGBA to RGB (drop alpha)
    frame = frame[..., :3]
    frames.append(frame)
    plt.close(fig)

# Save as GIF
imageio.mimsave('results/animations/oscillator_positions_overlay_anim.gif', frames, duration=1/25, loop=0)
print("GIF saved as 'oscillator_positions_overlay_anim.gif'.")


### Oscillator side-by-side comparison

In [ ]:

# Number of consecutive states to show in the animation (e.g., 24, up to 200)
gif_n = min(200, o_batch.size(0))

# Choose a random starting point for the sequence
gif_start_idx = np.random.randint(0, o_batch.size(0) - gif_n + 1)
gif_o_seq = o_batch[gif_start_idx:gif_start_idx + gif_n]
gif_actuator_seq = actuator_inputs[gif_start_idx:gif_start_idx + gif_n]

with torch.no_grad():
    # Encode all observations to get their latent means (mu)
    _, _, mu_seq, logvar_seq = vae(gif_o_seq)

    # For this visualization, use the true latent means (mu_seq) for oscillator positions,
    # not the predicted rollout, to avoid drift/accumulation artifacts.
    gif_z_for_positions = mu_seq

    # For comparison, get the original observations as numpy
    gif_orig_o = gif_o_seq.cpu().numpy()

    # Get oscillator positions for the sequence (using true mu_seq, not rollout)
    if config["dynamics_type"]=="harmonic2d_proj":
        gif_oscillator_positions = vae.decoder.project_oscillator_position_and_velocity(gif_z_for_positions)  # shape: (gif_n, num_osc, 2)
    elif config["dynamics_type"]=="harmonic2d_vel":
        gif_oscillator_positions = vae.decoder.get_attention_weights(gif_z_for_positions,return_peak_location=True)[1]

# Prepare for animation: left = oscillator positions on image, right = latent points
is_color = gif_orig_o.shape[1] == 3 or gif_orig_o.shape[1] == 4

# Get latent points for right plot
gif_z_np = gif_z_for_positions.detach().cpu().numpy()  # shape: (gif_n, latent_dim)
if gif_z_np.ndim == 2 and gif_z_np.shape[1] % 2 == 0:
    n_nodes = gif_oscillator_positions.shape[1]
    gif_z_points = gif_z_np.reshape(gif_n, n_nodes, -1)  # (gif_n, n_nodes, 2) if latent_dim==2*n_nodes
else:
    # fallback: treat as (gif_n, latent_dim)
    gif_z_points = gif_z_np
    n_nodes = gif_z_points.shape[1] if gif_z_points.ndim == 2 else 1

# Choose a colormap and assign a color to each oscillator/latent
cmap = plt.get_cmap('tab10' if n_nodes <= 10 else 'tab20')
osc_colors = [cmap(i % cmap.N) for i in range(n_nodes)]

# Set up the figure and axes
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
ax_img, ax_latent = axes

# --- Left: Oscillator positions overlay on image ---
if is_color:
    img = np.transpose(gif_orig_o[0], (1, 2, 0))
    if img.shape[2] == 4:
        img = img[..., :3]
else:
    img = gif_orig_o[0][0]
    img = np.stack([img]*3, axis=-1)
img_min, img_max = img.min(), img.max()
img_norm = (img - img_min) / (img_max - img_min + 1e-8) if img_max > img_min else np.zeros_like(img)
img_h, img_w = img.shape[:2]
im_artist = ax_img.imshow(img_norm, alpha=0.9)

# Instead of a single scatter, use one scatter per oscillator for color
osc_scatters = []
for j in range(n_nodes):
    sc = ax_img.scatter([], [], s=100, c=[osc_colors[j]], edgecolors='k', linewidths=1.0, alpha=0.9)
    osc_scatters.append(sc)
ax_img.set_title('Oscillator Positions Overlay')
ax_img.axis('off')

# --- Right: Latent points ---
if gif_z_points.ndim == 3 and gif_z_points.shape[2] >= 2:
    # Only plot first two dims if more
    latent_scatters = []
    for j in range(n_nodes):
        sc = ax_latent.scatter([], [], s=100, c=[osc_colors[j]], edgecolors='k', linewidths=1.0, alpha=0.9)
        latent_scatters.append(sc)
    ax_latent.set_xlim(np.min(gif_z_points[...,0])-0.1, np.max(gif_z_points[...,0])+0.1)
    ax_latent.set_ylim(np.min(gif_z_points[...,1])-0.1, np.max(gif_z_points[...,1])+0.1)
    ax_latent.set_xlabel('z[0]')
    ax_latent.set_ylabel('z[1]')
    ax_latent.set_title('Latent Points (per oscillator)')
else:
    # fallback: plot all dims as a line
    latent_line, = ax_latent.plot([], [], 'o-', color='red')
    ax_latent.set_xlim(0, gif_z_points.shape[1]-1)
    ax_latent.set_ylim(np.min(gif_z_points)-0.1, np.max(gif_z_points)+0.1)
    ax_latent.set_xlabel('Latent dim')
    ax_latent.set_ylabel('Value')
    ax_latent.set_title('Latent Vector')

plt.tight_layout()

def init():
    # Left
    for sc in osc_scatters:
        sc.set_offsets(np.zeros((0,2)))
    # Right
    if gif_z_points.ndim == 3 and gif_z_points.shape[2] >= 2:
        for sc in latent_scatters:
            sc.set_offsets(np.zeros((0,2)))
        return tuple(osc_scatters + latent_scatters)
    else:
        latent_line.set_data([], [])
        return tuple(osc_scatters + [latent_line])

def animate(i):
    # --- Left: Oscillator positions overlay on image ---
    if is_color:
        img = np.transpose(gif_orig_o[i], (1, 2, 0))
        if img.shape[2] == 4:
            img = img[..., :3]
    else:
        img = gif_orig_o[i][0]
        img = np.stack([img]*3, axis=-1)
    img_min, img_max = img.min(), img.max()
    img_norm = (img - img_min) / (img_max - img_min + 1e-8) if img_max > img_min else np.zeros_like(img)
    im_artist.set_data(img_norm)

    pos = gif_oscillator_positions[i]
    if isinstance(pos, torch.Tensor):
        pos = pos.detach().cpu().numpy()
    # pos shape: (num_osc, 2), values in [-1, 1]
    y_norm, x_norm = pos[:,0], pos[:,1]
    y_pix = ((y_norm + 1) / 2) * (img_h - 1)
    x_pix = ((x_norm + 1) / 2) * (img_w - 1)
    for j, sc in enumerate(osc_scatters):
        if j < len(x_pix):
            sc.set_offsets(np.array([[x_pix[j], y_pix[j]]]))
        else:
            sc.set_offsets(np.zeros((0,2)))

    # --- Right: Latent points ---
    if gif_z_points.ndim == 3 and gif_z_points.shape[2] >= 2:
        for j, sc in enumerate(latent_scatters):
            if j < gif_z_points.shape[1]:
                sc.set_offsets(gif_z_points[i, j, :2][None, :])
            else:
                sc.set_offsets(np.zeros((0,2)))
        return tuple(osc_scatters + latent_scatters)
    else:
        latent_line.set_data(np.arange(gif_z_points.shape[1]), gif_z_points[i])
        return tuple(osc_scatters + [latent_line])

ani = animation.FuncAnimation(
    fig, animate, frames=gif_n, init_func=init, blit=True, interval=40
)

plt.close(fig)  # Prevents duplicate static plot in notebook

# To save as mp4 (optional, uncomment if needed)
ani.save('results/animations/oscillator_positions_and_latents.mp4', writer='ffmpeg', fps=25)


### Multi-step prediction gif

In [ ]:
gif_n = min(60, o_batch.size(0))

gif_start_idx = np.random.randint(0, o_batch.size(0) - gif_n + 1)
gif_o_seq = o_batch[gif_start_idx:gif_start_idx + gif_n]
gif_actuator_seq = actuator_inputs[gif_start_idx:gif_start_idx + gif_n]

with torch.no_grad():
    if config["use_vae"]:
        _, _, mu_seq, logvar_seq = vae(gif_o_seq)
    else:
        _, mu_seq = vae(gif_o_seq)

    delta_t = torch.tensor(config["delta_t"], device=gif_o_seq.device, dtype=gif_o_seq.dtype)
    o_dot0 = (gif_o_seq[1:2] - gif_o_seq[0:1]) / delta_t
    z_dot0 = vae.latent_velocity_from_observation_velocity(gif_o_seq[0:1], o_dot0)

    z_current = mu_seq[0:1]
    z_dot_current = z_dot0

    gif_z_pred_seq = [z_current.squeeze(0)]
    gif_z_dot_pred_seq = [z_dot_current.squeeze(0)]
    
    for t in range(gif_n-1):
        z_next, z_dot_next = latent_dynamics(
            z_current,
            z_dot_current,
            gif_actuator_seq[t].unsqueeze(0),
            dt=config["delta_t"]
        )
        
        z_current = z_next
        z_dot_current = z_dot_next
        
        gif_z_pred_seq.append(z_next.squeeze(0))
        gif_z_dot_pred_seq.append(z_dot_next.squeeze(0))

    gif_z_pred_seq = torch.stack(gif_z_pred_seq, dim=0)
    gif_z_dot_pred_seq = torch.stack(gif_z_dot_pred_seq, dim=0)

    gif_recon_o_seq = vae.decode(gif_z_pred_seq)
    gif_orig_o = gif_o_seq.cpu().numpy()
    gif_recon_o = gif_recon_o_seq.cpu().numpy()

is_color = gif_orig_o.shape[1] == 3 or gif_orig_o.shape[1] == 4

frames = []
for i in range(gif_n):
    if is_color:
        orig_img = np.transpose(gif_orig_o[i], (1, 2, 0))
        recon_img = np.transpose(gif_recon_o[i], (1, 2, 0))
        if orig_img.shape[2] == 4:
            orig_img = orig_img[..., :3]
        if recon_img.shape[2] == 4:
            recon_img = recon_img[..., :3]
        orig_img_uint8 = (orig_img * 255).clip(0,255).astype('uint8')
        recon_img_uint8 = (recon_img * 255).clip(0,255).astype('uint8')
    else:
        orig_img = gif_orig_o[i][0]
        recon_img = gif_recon_o[i][0]
        orig_img_uint8 = (orig_img * 255).clip(0,255).astype('uint8')
        recon_img_uint8 = (recon_img * 255).clip(0,255).astype('uint8')
        orig_img_uint8 = np.stack([orig_img_uint8]*3, axis=-1)
        recon_img_uint8 = np.stack([recon_img_uint8]*3, axis=-1)

    side_by_side = np.concatenate([orig_img_uint8, recon_img_uint8], axis=1)

    frame_bgr = cv2.cvtColor(side_by_side, cv2.COLOR_RGB2BGR)
    text = f"{i+1}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.3
    color = (0, 255, 0)
    thickness = 1
    org = (0, 8)
    cv2.putText(frame_bgr, text, org, font, font_scale, color, thickness, cv2.LINE_AA)
    side_by_side_rgb_with_text = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    frames.append(side_by_side_rgb_with_text)

imageio.mimsave('results/animations/original_vs_predicted.gif', frames, duration=1/25, loop=0)
print("GIF saved as 'original_vs_predicted.gif'.")

### Multi-step prediction plot

In [ ]:
T, latent_dim = mu_seq.shape

plt.figure(figsize=(16, 2.5 * latent_dim))
for i in range(latent_dim):
    plt.subplot(latent_dim, 1, i+1)
    plt.plot(mu_seq[:, i].cpu().numpy(), label='True $z_{}$'.format(i+1))
    plt.plot(gif_z_pred_seq[:, i].cpu().numpy(), label='Predicted $z_{}$'.format(i+1), linestyle='--')
    plt.ylabel(f'$z_{{{i+1}}}$')
    plt.legend()
    if i == 0:
        plt.title("True vs Predicted Latent Variables ($z$) per Dimension")
    if i == latent_dim - 1:
        plt.xlabel("Time step")
plt.tight_layout()
plt.show()

delta_t = config["delta_t"]

mu_seq_np = mu_seq.cpu().numpy()
mu_dot_seq = np.zeros_like(mu_seq_np)
mu_dot_seq[0] = (mu_seq_np[1] - mu_seq_np[0]) / delta_t
if T > 2:
    mu_dot_seq[1:-1] = (mu_seq_np[2:] - mu_seq_np[:-2]) / (2.0 * delta_t)
mu_dot_seq[-1] = (mu_seq_np[-1] - mu_seq_np[-2]) / delta_t

gif_z_dot_pred_seq_np = gif_z_dot_pred_seq.cpu().numpy()

plt.figure(figsize=(16, 2.5 * latent_dim))
for i in range(latent_dim):
    plt.subplot(latent_dim, 1, i+1)
    plt.plot(mu_dot_seq[:, i], label='True $\dot{{z}}_{}$'.format(i+1))
    plt.plot(gif_z_dot_pred_seq_np[:, i], label='Predicted $\dot{{z}}_{}$'.format(i+1), linestyle='--')
    plt.ylabel(f'$\dot{{z}}_{{{i+1}}}$')
    plt.legend()
    if i == 0:
        plt.title("True vs Predicted Latent Velocities ($\dot{{z}}$) per Dimension")
    if i == latent_dim - 1:
        plt.xlabel("Time step")
plt.tight_layout()
plt.show()